In [1]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

RAW = Path('raw')
CLEAN = Path('cleaned')
CLEAN.mkdir(exist_ok=True)

sellers = pd.read_csv(RAW/ 'olist_sellers_dataset.csv')

In [2]:
sellers.info()

<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   int64
 2   seller_city             3095 non-null   str  
 3   seller_state            3095 non-null   str  
dtypes: int64(1), str(3)
memory usage: 96.8 KB


In [3]:
sellers.isna().sum()

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

In [4]:
sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [5]:
sellers['seller_id'].is_unique

True

In [6]:
sellers['seller_state'].nunique()

23

In [10]:
(sellers['seller_zip_code_prefix'].astype(str).str.len() ==4).sum()

np.int64(1027)

In [11]:
(sellers['seller_zip_code_prefix'].astype(str).str.len()).value_counts()

seller_zip_code_prefix
5    2068
4    1027
Name: count, dtype: int64

In [15]:
sellers['seller_zip_code_prefix'] = sellers['seller_zip_code_prefix'].astype(str).str.zfill(5)

In [16]:
(sellers['seller_zip_code_prefix'].astype(str).str.len()).value_counts()

seller_zip_code_prefix
5    3095
Name: count, dtype: int64

In [17]:
sellers['seller_state'].nunique()

23

In [18]:
sellers['seller_state'].value_counts()

seller_state
SP    1849
PR     349
MG     244
SC     190
RJ     171
RS     129
GO      40
DF      30
ES      23
BA      19
CE      13
PE       9
PB       6
RN       5
MS       5
MT       4
RO       2
SE       2
AC       1
PI       1
MA       1
AM       1
PA       1
Name: count, dtype: int64

In [19]:
(sellers['seller_state'] != sellers['seller_state'].str.strip()).sum()

np.int64(0)

In [21]:
sellers['seller_city'].nunique()

611

In [29]:
odd = sellers.loc[sellers['seller_city'].str.contains(r"[^a-z \-']", regex = True), 'seller_city']
odd.shape[0]

24

In [33]:
odd.unique()

<StringArray>
[                            'auriflama/sp',
                    'sao paulo / sao paulo',
                               'são paulo',
                    'santa barbara d´oeste',
                                 '04482255',
 'novo hamburgo, rio grande do sul, brasil',
                           'cariacica / es',
                                   'sbc/sp',
           'arraial d'ajuda (porto seguro)',
                    'santo andre/sao paulo',
                                  'sp / sp',
                           'maua/sao paulo',
                     'mogi das cruzes / sp',
           'rio de janeiro \rio de janeiro',
                  'barbacena/ minas gerais',
          'rio de janeiro / rio de janeiro',
                               'pinhais/pr',
               'ribeirao preto / sao paulo',
                  'carapicuiba / sao paulo',
                'vendas@creditparts.com.br',
                'sao sebastiao da grama/sp',
   'rio de janeiro, rio de janeiro, brasi

In [34]:
risky = ['sbc/sp', 'sp / sp', '04482255', 'vendas@creditparts.com.br']
print(sellers.loc[sellers['seller_city'].isin(risky),
                  ['seller_zip_code_prefix', 'seller_city', 'seller_state']].to_string())

     seller_zip_code_prefix                seller_city seller_state
517                   22790                   04482255           RJ
869                   09726                     sbc/sp           SP
1004                  03363                    sp / sp           SP
2258                  87025  vendas@creditparts.com.br           PR


In [35]:
c = sellers['seller_city'].str.replace('´', "'", regex=False)                       # odd apostrophe -> normal one
c = c.str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('ascii')  # são -> sao
c = c.str.split(r'[/\\,(]', regex=True).str[0].str.strip()                          # keep text before / \ , (

sellers['city_was_fixed'] = c != sellers['seller_city']
sellers['seller_city'] = c

print(sellers['city_was_fixed'].sum())
print(sellers.loc[sellers['city_was_fixed'], 'seller_city'].to_string())

22
237                  auriflama
246                  sao paulo
360                  sao paulo
476      santa barbara d'oeste
551              novo hamburgo
622                  cariacica
869                        sbc
874            arraial d'ajuda
945                santo andre
1004                        sp
1159                      maua
1337           mogi das cruzes
1346            rio de janeiro
1447                 barbacena
1649            rio de janeiro
1712                   pinhais
1920            ribeirao preto
2162               carapicuiba
2340     santa barbara d'oeste
2620    sao sebastiao da grama
2988            rio de janeiro
3015                   jacarei


In [36]:
risky = ['sbc/sp', 'sp / sp', '04482255', 'vendas@creditparts.com.br']
print(sellers.loc[sellers['seller_city'].isin(risky),
                  ['seller_zip_code_prefix', 'seller_city', 'seller_state']].to_string())

     seller_zip_code_prefix                seller_city seller_state
517                   22790                   04482255           RJ
2258                  87025  vendas@creditparts.com.br           PR


In [37]:
risky = ['sbc', 'sp', '04482255', 'vendas@creditparts.com.br']
print(sellers.loc[sellers['seller_city'].isin(risky),
                  ['seller_zip_code_prefix', 'seller_city', 'seller_state']].to_string())

     seller_zip_code_prefix                seller_city seller_state
517                   22790                   04482255           RJ
869                   09726                        sbc           SP
1004                  03363                         sp           SP
1117                  12903                         sp           SP
1246                  04776                         sp           SP
1814                  09861                        sbc           SP
1889                  16021                         sp           SP
2258                  87025  vendas@creditparts.com.br           PR
2536                  05141                         sp           SP


In [38]:
print((sellers['seller_zip_code_prefix'].astype(str).str.len() == 4).sum())
print(sellers['seller_state'].nunique())

0
23


In [39]:
fix_zips = ['22790', '09726', '09861', '03363', '12903', '04776', '16021', '05141', '87025']
bad = ['sbc', 'sp', '04482255', 'vendas@creditparts.com.br']

known = sellers.loc[
    sellers['seller_zip_code_prefix'].astype(str).str.zfill(5).isin(fix_zips)
    & ~sellers['seller_city'].isin(bad),
    ['seller_zip_code_prefix', 'seller_city', 'seller_state']
]
print(known.sort_values('seller_zip_code_prefix').to_string())

     seller_zip_code_prefix     seller_city seller_state
2573                  05141        pirituba           SP
222                   22790  rio de janeiro           RJ
770                   22790  rio de janeiro           RJ
1543                  22790  rio de janeiro           RJ
2703                  22790  rio de janeiro           RJ
809                   87025         maringa           PR
1954                  87025         maringa           PR
2357                  87025         maringa           PR


In [40]:
print(sellers['seller_zip_code_prefix'].dtype)
print(sellers['seller_zip_code_prefix'].astype(str).str.len().value_counts())

str
seller_zip_code_prefix
5    3095
Name: count, dtype: int64


In [41]:
bad = ['sp', '04482255', 'vendas@creditparts.com.br']

# 1. documented manual mapping
sbc_mask = sellers['seller_city'] == 'sbc'
sellers.loc[sbc_mask, 'seller_city'] = 'sao bernardo do campo'

# 2. lookup: city of other sellers who share the same zip
is_bad = sellers['seller_city'].isin(bad)
zip_city = (sellers.loc[~is_bad & ~sbc_mask]
            .groupby('seller_zip_code_prefix')['seller_city']
            .agg(lambda s: s.mode().iloc[0]))
sellers.loc[is_bad, 'seller_city'] = sellers.loc[is_bad, 'seller_zip_code_prefix'].map(zip_city)

# 3. no match -> unknown
sellers['seller_city'] = sellers['seller_city'].fillna('unknown')

# 4. flag everything touched
sellers['city_was_fixed'] = sellers['city_was_fixed'] | is_bad | sbc_mask

print(sellers['city_was_fixed'].sum())
print(sellers.loc[is_bad | sbc_mask, ['seller_zip_code_prefix', 'seller_city', 'seller_state']].to_string())

29
     seller_zip_code_prefix            seller_city seller_state
517                   22790         rio de janeiro           RJ
869                   09726  sao bernardo do campo           SP
1004                  03363                unknown           SP
1117                  12903                unknown           SP
1246                  04776                unknown           SP
1814                  09861  sao bernardo do campo           SP
1889                  16021                unknown           SP
2258                  87025                maringa           PR
2536                  05141               pirituba           SP


In [42]:
sellers['seller_city'] = sellers['seller_city'].str.title()
print(sellers['seller_city'].nunique())
print(sellers['seller_zip_code_prefix'].str.len().eq(5).all())

items = pd.read_csv(CLEAN/'order_items_clean.csv', usecols=['order_id', 'seller_id'])
print((~items['seller_id'].isin(sellers['seller_id'])).sum())    # items from unknown sellers
print((~sellers['seller_id'].isin(items['seller_id'])).sum())    # sellers that never sold

589
True
0
0


In [43]:
sellers.to_csv(CLEAN/'sellers_clean.csv', index=False)

check = pd.read_csv(CLEAN/'sellers_clean.csv', dtype={'seller_zip_code_prefix': str})
print(check.shape)
print(check['seller_id'].is_unique)
print(check['seller_zip_code_prefix'].str.len().value_counts())
print(check['seller_city'].nunique())
print(check['city_was_fixed'].sum())

(3095, 5)
True
seller_zip_code_prefix
5    3095
Name: count, dtype: int64
589
29


In [44]:
sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state,city_was_fixed
0,3442f8959a84dea7ee197c632cb2df15,13023,Campinas,SP,False
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,Mogi Guacu,SP,False
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,Rio De Janeiro,RJ,False
3,c0f3eea2e14555b6faeea3dd58c1b1c3,04195,Sao Paulo,SP,False
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,Braganca Paulista,SP,False
